In [ ]:
!pip -q install sentence-transformers datasets scikit-learn joblib fastapi uvicorn

In [ ]:
%%writefile features.py
import re, math, numpy as np
FRESH  = re.compile(r"\b(latest|deprecated|release notes|changelog|still (supported|maintained)|new in|what changed|breaking changes?)\b|\bv?\d+\.\d+(\.\d+)?\b", re.I)
TRACE  = re.compile(r"traceback|exception|error:|stack trace|segfault|at line \d+|\bpanic\b|undefined is not|panicked", re.I)
DESIGN = re.compile(r"architecture|scal(e|ab)|trade-?offs?|system design|microservice|multi-tenant|throughput|high availability", re.I)
SHELL  = re.compile(r"\b(regex|sql|bash|shell|awk|sed|cron|dockerfile|docker|curl|git)\b", re.I)
CODEKW = re.compile(r"\b(def|class|function|import|return|const|let|var|public|void|SELECT|FROM)\b")
REFAC  = re.compile(r"\b(refactor|clean up|review|optimi[sz]e|rewrite|simplify)\b", re.I)

def hand_features(p):
    words = len(p.split())
    return [math.log1p(words), math.log1p(p.count("\n")), float("```" in p),
            float(bool(TRACE.search(p))), float(bool(FRESH.search(p))),
            float(bool(DESIGN.search(p))), float(bool(SHELL.search(p))),
            float(bool(CODEKW.search(p))), float(bool(REFAC.search(p))),
            p.count("?")/3.0]

def featurize(texts, encoder):
    emb = encoder.encode(list(texts), batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    hand = np.array([hand_features(t) for t in texts])
    return np.hstack([emb, hand])

In [ ]:
import random, re, pandas as pd
from datasets import load_dataset
random.seed(0)
rows = []   # (text, task, tier, source)

def add(texts, task, tier, src):
    for t in texts:
        t = (t or "").strip()
        if 8 < len(t) < 2500: rows.append((t, task, tier, src))

def safe(fn, name):
    try: fn(); print("ok  ", name)
    except Exception as e: print("SKIP", name, "->", str(e)[:120])

# --- real datasets ---
def mbpp():
    d = load_dataset("mbpp", "full", split="train")
    add(d["text"], "generate", "cheap", "mbpp")
def codealpaca():
    d = load_dataset("sahil2801/CodeAlpaca-20k", split="train")
    for r in d.select(range(min(4000, len(d)))):
        t = r["instruction"] + (("\n" + r["input"]) if r.get("input") else "")
        w = len(t.split())
        if re.search(r"refactor|optimi[sz]e|improve|rewrite|simplify", t, re.I): add([t], "refactor", "cheap" if w < 40 else "expensive", "codealpaca")
        elif re.search(r"\b(fix|bug|error|debug)\b", t, re.I): add([t], "debug", "cheap", "codealpaca")
        elif random.random() < 0.25: add([t], "generate", "cheap" if w < 40 else "expensive", "codealpaca")
def magicoder():
    d = load_dataset("ise-uiuc/Magicoder-OSS-Instruct-75K", split="train").shuffle(seed=0).select(range(600))
    add(d["problem"], "generate", "expensive", "magicoder")
def sqlctx():
    d = load_dataset("b-mc2/sql-create-context", split="train").shuffle(seed=0).select(range(400))
    add(d["question"], "shell", "cheap", "sql-create-context")
def dolly():
    d = load_dataset("databricks/databricks-dolly-15k", split="train")
    sel = [r["instruction"] for r in d if r["category"] in ("open_qa","general_qa","closed_qa") and len(r["instruction"].split()) < 25]
    random.shuffle(sel); add(sel[:300], "explain", "cheap", "dolly")

for f in (mbpp, codealpaca, magicoder, sqlctx, dolly): safe(f, f.__name__)

# --- templates: debug / design / docs / refactor / explain / shell (no dataset covers these) ---
ERR = ["KeyError: 'user_id'", "TypeError: 'NoneType' object is not subscriptable", "ValueError: invalid literal for int() with base 10",
       "TypeError: Cannot read properties of undefined (reading 'map')", "java.lang.NullPointerException", "panic: runtime error: index out of range [3] with length 3",
       "ModuleNotFoundError: No module named 'requests'", "CORS error: No 'Access-Control-Allow-Origin' header", "SegmentationFault (core dumped)", "Uncaught Error: Too many re-renders"]
FILES = ["app.py","views.py","index.js","UserService.java","main.go","App.jsx","worker.py","server.ts"]
QS = ["Why does this happen intermittently under load?", "It only fails in production, not locally. What could be different?", "How do I fix this?", "What is the root cause here?"]
for _ in range(160):
    e, f, n = random.choice(ERR), random.choice(FILES), random.randint(8, 400)
    add([f'Traceback (most recent call last):\n  File "{f}", line {n}, in handle\n    result = process(data)\n  File "{f}", line {n-5}, in process\n    {e}\n{random.choice(QS)}'], "debug", "expensive", "template")
for _ in range(80):
    e = random.choice(ERR)
    add([random.choice([f"why am I getting {e}", f"what does '{e}' mean?", f"how to fix {e}", f"quick fix for {e}"])], "debug", "cheap", "template")

SYS = ["multi-tenant job queue","real-time chat backend","rate limiter for a public API","URL shortener","notification service","payments ledger","feature flag service","log ingestion pipeline"]
REQ = ["scales to 10k requests per second","has to survive a regional outage","needs exactly-once processing","serves 5M daily users","must stay under 100ms p99 latency"]
PAIR = [("Kafka","SQS"),("Postgres","DynamoDB"),("Redis","Memcached"),("REST","gRPC"),("monolith","microservices")]
for _ in range(120):
    s, r, (a, b) = random.choice(SYS), random.choice(REQ), random.choice(PAIR)
    add([random.choice([f"Design a {s} that {r}. Compare {a} vs {b} and explain the tradeoffs.", f"How should I architect a {s}? It {r}.", f"System design: {s}, {r}. What are the failure modes?"])], "design", "expensive", "template")

LIB = ["React","Next.js","Pydantic","FastAPI","Django","Node.js","TypeScript","Tailwind","LangChain","Spring Boot","PyTorch","Vite"]
for _ in range(140):
    l, v = random.choice(LIB), f"{random.randint(1,19)}.{random.randint(0,9)}"
    add([random.choice([f"what changed in {l} {v}?", f"is the old API deprecated in {l} {v}?", f"latest version of {l} and its breaking changes", f"does {l} {v} still support the previous config format?", f"{l} release notes {v}", f"is {l} still maintained?"])], "docs", "cheap", "template")

SNIP = ["def f(x):\n    if x:\n        if x > 0:\n            if x < 10:\n                return True\n    return False", "for i in range(len(a)):\n    for j in range(len(a)):\n        if a[i] == a[j] and i != j: out.append(a[i])"]
for _ in range(70):
    add([f"Refactor this to be more readable and efficient:\n```python\n{random.choice(SNIP)}\n```"], "refactor", "expensive", "template")
for _ in range(40):
    add([random.choice(["how can I clean up nested ifs?","review my variable names","simplify this loop","is there a cleaner way to write a null check?"])], "refactor", "cheap", "template")

CON = ["git rebase","a Python decorator","async/await","a closure","dependency injection","a mutex","CORS","a Docker volume","big-O notation","a REST API"]
for _ in range(100):
    c = random.choice(CON)
    add([random.choice([f"what does {c} do?", f"explain {c} simply", f"hi, what is {c}?"])], "explain", "cheap", "template")
for _ in range(50):
    c = random.choice(CON)
    add([f"Explain in depth how {c} works internally, cover edge cases and failure modes, and show how it interacts with concurrency."], "explain", "expensive", "template")

TASKS = ["find files larger than 100MB","kill the process on port 3000","match an email address","rename all .txt files to .md","undo the last git commit","count lines in all python files","parse a date with regex","delete merged git branches"]
for _ in range(100):
    t = random.choice(TASKS)
    add([random.choice([f"command to {t}", f"bash one-liner to {t}", f"how do I {t}", f"regex or shell to {t}"])], "shell", "cheap", "template")

df = pd.DataFrame(rows, columns=["text","task","tier","source"]).drop_duplicates("text")
print(len(df)); print(df.groupby(["task","tier"]).size().unstack(fill_value=0))

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
from features import featurize

CAP = 400
df = df.groupby("task", group_keys=False).apply(lambda g: g.sample(min(len(g), CAP), random_state=0)).reset_index(drop=True)
enc = SentenceTransformer("all-MiniLM-L6-v2")
X = featurize(df.text.tolist(), enc)
idx_tr, idx_te = train_test_split(np.arange(len(df)), test_size=0.2, stratify=df.task, random_state=0)
task_clf = LogisticRegression(max_iter=3000, C=2.0, class_weight="balanced").fit(X[idx_tr], df.task.iloc[idx_tr])
tier_clf = LogisticRegression(max_iter=3000, C=2.0, class_weight="balanced").fit(X[idx_tr], df.tier.iloc[idx_tr])
print("TASK"); print(classification_report(df.task.iloc[idx_te], task_clf.predict(X[idx_te])))
print("TIER"); print(classification_report(df.tier.iloc[idx_te], tier_clf.predict(X[idx_te])))
print(pd.crosstab(df.task.iloc[idx_te], task_clf.predict(X[idx_te])))

In [ ]:
mine = [
 ("my flask app throws 500 on POST /login but works in postman, here is the log: KeyError 'csrf_token'", "debug", "cheap"),
 ("write a python script that watches a folder and uploads new files to s3 with retries and logging", "generate", "expensive"),
 ("whats the difference between let and const", "explain", "cheap"),
 ("regex for indian phone numbers", "shell", "cheap"),
 ("is pydantic v3 out yet and what broke", "docs", "cheap"),
 ("how would you design a collaborative editor like google docs, conflict resolution, offline mode", "design", "expensive"),
 ("make this function faster:\n```python\ndef f(a):\n    return [x for x in a if x in a]\n```", "refactor", "expensive"),
 ("kill process on port 8080", "shell", "cheap"),
 ("write a function to check palindrome", "generate", "cheap"),
 ("latest next.js version and does app router still need use client", "docs", "cheap"),
]
Xm = featurize([m[0] for m in mine], enc)
pt, pr = task_clf.predict(Xm), tier_clf.predict(Xm)
for (t, tk, tr), a, b in zip(mine, pt, pr):
    print("OK " if (a==tk and b==tr) else "MISS", f"task {tk}->{a} | tier {tr}->{b} | {t[:60]!r}")
print("task acc", np.mean([a==m[1] for a,m in zip(pt,mine)]), "tier acc", np.mean([b==m[2] for b,m in zip(pr,mine)]))

In [ ]:
import joblib
task_clf = LogisticRegression(max_iter=3000, C=2.0, class_weight="balanced").fit(X, df.task)
tier_clf = LogisticRegression(max_iter=3000, C=2.0, class_weight="balanced").fit(X, df.tier)
joblib.dump({"task": task_clf, "tier": tier_clf}, "router.joblib")
print("saved router.joblib")

In [ ]:
%%writefile serve.py
import joblib
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer
from features import featurize, hand_features

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
enc = SentenceTransformer("all-MiniLM-L6-v2")
m = joblib.load("router.joblib")

class Req(BaseModel):
    prompt: str

@app.post("/predict")
def predict(r: Req):
    x = featurize([r.prompt], enc)
    tp, rp = m["task"].predict_proba(x)[0], m["tier"].predict_proba(x)[0]
    task, tier = m["task"].classes_[tp.argmax()], m["tier"].classes_[rp.argmax()]
    hf = hand_features(r.prompt); why = []
    if hf[3]: why.append("Contains an error or stack trace")
    if hf[4]: why.append("Mentions versions or recent changes")
    if hf[5]: why.append("Architecture or tradeoff language")
    if hf[2] or hf[1] > 1.5: why.append("Includes pasted code or multiple lines")
    why.append(f"Model: task {task} ({tp.max():.0%}), tier {tier} ({rp.max():.0%})")
    return {"task": task, "tier": tier, "confidence": float((tp.max() + rp.max()) / 2), "reasons": why}